# Direct GAE Training and Evaluation — Google Colab

Single-run debugger for one frozen `graph_dataset.pt`. It does not invoke parsing, enrichment, sequencing, or dataset preparation.

For the HDFS **ablation campaign** (Family A/B, per-run eval packs, leaderboard) use [`7_AblationStudy_Colab.ipynb`](./7_AblationStudy_Colab.ipynb) and `notebooks/ABLATION.md`. Leave `RUN_EXPERIMENT_MATRIX` off unless you are poking at one graph.

## 1 — Direct Training Configuration

Set the prepared graph-bundle path and the local/Drive locations. The graph bundle is the only dataset input.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import gzip
import json
import os
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
REPOSITORY_URL = "https://github.com/michaail/hybrid-logs-analyzer-research.git"
GIT_REF = "ablation-cursor"

DATASET = "hdfs"  # "hdfs" or "bgl"
GRAPH_DATASET_RELATIVE_PATH = (
    "data/processed/hdfs/"
    "20260818_0002_1_parser_3_graph_dataset.pt.gz"
)
RUN_ID_PREFIX = f"{DATASET}_gae_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
RUN_EXPERIMENT_MATRIX = False
REUSE_LOCAL_GRAPH_FILE = True


def find_local_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "modules" / "models" / "gae.py").exists():
            return candidate
    raise FileNotFoundError("Run the notebook from the project repository or use Google Colab.")


REPO_ROOT = Path("/content/hybrid-logs-analyzer-research") if IN_COLAB else find_local_repository_root()
WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else REPO_ROOT
DRIVE_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")
EXPERIMENT_ROOT = WORKSPACE_ROOT / "outputs" / DATASET / "gae_training_notebook"
print(f"Running in Google Colab: {IN_COLAB}")

Running in Google Colab: True


## 2 — Clone or Update the Repository

In [2]:
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ])
    else:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)
print(f"Code checkout: {REPO_ROOT}")
print(f"Workspace: {WORKSPACE_ROOT}")

Mounted at /content/drive
Code checkout: /content/hybrid-logs-analyzer-research
Workspace: /content/workspace


## 3 — Install Colab-Compatible Dependencies

In [3]:
if IN_COLAB:
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "scripts" / "install_colab.py"),
        "--project-root", str(REPO_ROOT),
    ])
else:
    print("Local execution: use the project's virtual environment and requirements.txt.")

## 4 — Configure Python Paths and Verify GPU Availability

In [4]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

gae_module = REPO_ROOT / "src" / "modules" / "models" / "gae.py"
if not gae_module.exists():
    raise FileNotFoundError(
        f"Missing {gae_module}. This Colab clone of {GIT_REF!r} does not contain "
        "the GAE package. Commit and push src/modules/models/, then "
        "Runtime → Restart session and rerun from the clone cell."
    )

import torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GiB")
elif IN_COLAB:
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU, then restart.")

Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA: True | devices: 1
GPU: Tesla T4
Available GPU memory: 14.5 GiB


## 5 — No External-Service Setup Required

Direct training from the prepared graph dataset has no parser, LLM, embedding-model, or Azure OpenAI dependency.

In [5]:
print(
    "Direct mode: the prepared graph bundle already contains graphs, split indices, "
    "node features, edge attributes, and labels used for validation/evaluation."
)

Direct mode: the prepared graph bundle already contains graphs, split indices, node features, edge attributes, and labels used for validation/evaluation.


## 6 — Stage the Prepared Graph Dataset from Drive

The compressed graph bundle is copied once to Colab-local storage and decompressed there. Training never reads the large data file from the Drive mount.

In [6]:
def stage_graph_dataset(relative_path: str) -> Path:
    """Copy a graph bundle from Drive once, then decompress it on local Colab storage."""
    if IN_COLAB:
        source = DRIVE_ROOT / relative_path
        if not source.exists():
            raise FileNotFoundError(f"Missing graph bundle in Drive: {source}")
        staged = WORKSPACE_ROOT / relative_path
        staged.parent.mkdir(parents=True, exist_ok=True)
        if not staged.exists() or not REUSE_LOCAL_GRAPH_FILE:
            print(f"Copying {source.name} to local storage...")
            shutil.copy2(source, staged)
    else:
        staged = WORKSPACE_ROOT / relative_path
        if not staged.exists():
            raise FileNotFoundError(f"Missing local graph bundle: {staged}")

    if staged.suffix != ".gz":
        return staged
    graph_path = staged.with_suffix("")
    if not graph_path.exists() or not REUSE_LOCAL_GRAPH_FILE:
        print(f"Decompressing {staged.name} → {graph_path.name}")
        with gzip.open(staged, "rb") as source, open(graph_path, "wb") as destination:
            shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
    return graph_path

GRAPH_DATASET_PATH = stage_graph_dataset(GRAPH_DATASET_RELATIVE_PATH)
print(f"Prepared graph dataset: {GRAPH_DATASET_PATH}")

Copying 20260818_0002_1_parser_3_graph_dataset.pt.gz to local storage...
Decompressing 20260818_0002_1_parser_3_graph_dataset.pt.gz → 20260818_0002_1_parser_3_graph_dataset.pt
Prepared graph dataset: /content/workspace/data/processed/hdfs/20260818_0002_1_parser_3_graph_dataset.pt


## 7 — Define Direct GAE Training Experiments

Each dictionary is a complete training configuration for the saved graph bundle. Enable the matrix to evaluate every configuration independently, or leave it disabled to run only the baseline.

## 8 — Validate Experiment Definitions and Load Graph Splits

In [7]:
EXPERIMENTS = [
    {
        "name": "baseline",
        "seed": 42,
        "train_mode": "clean",  # "clean" excludes anomalous training graphs; "noisy" uses all.
        "test_run": False,
        "test_samples": 5000,
        "hidden_dim": 128,
        "latent_dim": 64,
        "batch_size": 256,
        "epochs": 25,
        "learning_rate": 0.01 if DATASET == "hdfs" else 0.001,
        "alpha": 1.0,
        "beta": 1.0,
        "gamma": 1.0,
        "pre_normalize_edges": True,
        "minimum_edge_std": 0.1,
        "gine_aggregation": "sum",  # "sum", "mean", or "max"
        "node_transformation": "mlp",  # "mlp" or "linear"
    },
    {"name": "latent_32", "seed": 42, "latent_dim": 32},
    {"name": "hidden_256", "seed": 42, "hidden_dim": 256},
    {"name": "learning_rate_001", "seed": 42, "learning_rate": 0.001},
    {"name": "edge_weight_05", "seed": 42, "gamma": 0.5},
]
DEFAULT_EXPERIMENT = EXPERIMENTS[0]
active_experiments = []
for experiment in EXPERIMENTS if RUN_EXPERIMENT_MATRIX else EXPERIMENTS[:1]:
    merged = {**DEFAULT_EXPERIMENT, **experiment}
    active_experiments.append(merged)
print(f"Experiments selected: {[experiment['name'] for experiment in active_experiments]}")

Experiments selected: ['baseline']


In [8]:
import torch

REQUIRED_EXPERIMENT_KEYS = {
    "name", "seed", "train_mode", "test_run", "test_samples", "hidden_dim",
    "latent_dim", "batch_size", "epochs", "learning_rate", "alpha", "beta",
    "gamma", "pre_normalize_edges", "minimum_edge_std", "gine_aggregation",
    "node_transformation",
}
names = [experiment["name"] for experiment in active_experiments]
if len(names) != len(set(names)):
    raise ValueError("Experiment names must be unique.")
for experiment in active_experiments:
    missing = REQUIRED_EXPERIMENT_KEYS - set(experiment)
    if missing:
        raise ValueError(f"{experiment['name']} is missing: {sorted(missing)}")
    if experiment["train_mode"] not in {"clean", "noisy"}:
        raise ValueError("train_mode must be 'clean' or 'noisy'.")
    if experiment["gine_aggregation"] not in {"sum", "mean", "max"}:
        raise ValueError("gine_aggregation must be sum, mean, or max.")
    if experiment["node_transformation"] not in {"mlp", "linear"}:
        raise ValueError("node_transformation must be mlp or linear.")
    if any(float(experiment[key]) < 0 for key in ("alpha", "beta", "gamma")):
        raise ValueError("Loss weights must be non-negative.")
    if any(int(experiment[key]) <= 0 for key in ("hidden_dim", "latent_dim", "batch_size", "epochs", "test_samples")):
        raise ValueError("Model dimensions, batch size, epochs, and test_samples must be positive.")
    if float(experiment["learning_rate"]) <= 0 or float(experiment["minimum_edge_std"]) <= 0:
        raise ValueError("learning_rate and minimum_edge_std must be positive.")

bundle = torch.load(GRAPH_DATASET_PATH, map_location="cpu", weights_only=False)
required_bundle_keys = {"data_list", "idx_train", "idx_val", "idx_test", "node_dim", "edge_dim"}
missing_bundle_keys = required_bundle_keys - set(bundle)
if missing_bundle_keys:
    raise ValueError(f"Graph dataset is missing: {sorted(missing_bundle_keys)}")
print(f"Graphs: {len(bundle['data_list']):,}")
print(f"Node features: {bundle['node_dim']}; edge features: {bundle['edge_dim']}")
print(f"Saved splits — train: {len(bundle['idx_train']):,}, validation: {len(bundle['idx_val']):,}, test: {len(bundle['idx_test']):,}")

Graphs: 575,061
Node features: 530; edge features: 10
Saved splits — train: 402,542, validation: 86,259, test: 86,260


## 9 — Direct Training, Validation, and Test Evaluation

This section trains `AttributeAwareGAE` directly against the prepared graph bundle. It selects the best epoch by validation F1, then evaluates the selected checkpoint once on the held-out test split.

In [ ]:
import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from torch.optim import Adam
from torch_geometric.loader import DataLoader
from src.modules.models.gae import AttributeAwareGAE, compute_anomaly_scores, train_epoch
from src.modules.utils import seed_everything


def normalise_edges(train_graphs, val_graphs, test_graphs, *, enabled: bool, minimum_std: float):
    if not enabled:
        return None, None
    edges = [graph.edge_attr.float() for graph in train_graphs if getattr(graph, "edge_attr", None) is not None and graph.edge_attr.numel()]
    if not edges:
        return None, None
    stacked = torch.cat(edges, dim=0)
    mean = stacked.mean(dim=0)
    std = stacked.std(dim=0).clamp_min(minimum_std)
    for graph in [*train_graphs, *val_graphs, *test_graphs]:
        if getattr(graph, "edge_attr", None) is not None and graph.edge_attr.numel():
            graph.edge_attr = (graph.edge_attr.float() - mean) / std
    return mean.tolist(), std.tolist()


def score_metrics(labels, scores, threshold):
    predictions = (scores > threshold).astype(int)
    both_classes = len(np.unique(labels)) == 2
    return {
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "pr_auc": float(average_precision_score(labels, scores)) if both_classes else 0.0,
        "roc_auc": float(roc_auc_score(labels, scores)) if both_classes else 0.0,
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }


def select_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds):
        threshold = float(np.median(scores))
    else:
        f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1_values))])
    return threshold, score_metrics(labels, scores, threshold)


def train_direct(experiment):
    seed_everything(int(experiment["seed"]))
    graphs = [copy.deepcopy(graph) for graph in bundle["data_list"]]
    train_graphs = [graphs[int(index)] for index in bundle["idx_train"]]
    val_graphs = [graphs[int(index)] for index in bundle["idx_val"]]
    test_graphs = [graphs[int(index)] for index in bundle["idx_test"]]
    if experiment["train_mode"] == "clean":
        train_graphs = [graph for graph in train_graphs if int(graph.y.item()) == 0]
    if experiment["test_run"]:
        limit = int(experiment["test_samples"])
        train_graphs, val_graphs, test_graphs = train_graphs[:limit], val_graphs[:limit], test_graphs[:limit]
    if not train_graphs:
        raise ValueError("No graphs remain for training.")

    edge_mean, edge_std = normalise_edges(
        train_graphs, val_graphs, test_graphs,
        enabled=bool(experiment["pre_normalize_edges"]),
        minimum_std=float(experiment["minimum_edge_std"]),
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaders = {
        "train": DataLoader(train_graphs, batch_size=int(experiment["batch_size"]), shuffle=True),
        "val": DataLoader(val_graphs, batch_size=int(experiment["batch_size"])),
        "test": DataLoader(test_graphs, batch_size=int(experiment["batch_size"])),
    }
    model = AttributeAwareGAE(
        node_dim=int(bundle["node_dim"]), edge_dim=int(bundle["edge_dim"]),
        hidden_dim=int(experiment["hidden_dim"]), latent_dim=int(experiment["latent_dim"]),
        gine_aggregation=experiment["gine_aggregation"], node_transformation=experiment["node_transformation"],
    ).to(device)
    optimizer = Adam(model.parameters(), lr=float(experiment["learning_rate"]))
    weights = {key: float(experiment[key]) for key in ("alpha", "beta", "gamma")}
    history, best_state, best_threshold, best_f1 = [], None, 0.0, -1.0
    for epoch in range(1, int(experiment["epochs"]) + 1):
        total, structure, node, edge = train_epoch(model, loaders["train"], optimizer, device, **weights)
        val_scores, val_labels = compute_anomaly_scores(model, loaders["val"], device, **weights)
        threshold, val_metrics = select_threshold(val_labels, val_scores)
        history.append({
            "epoch": epoch, "train_total_loss": total, "train_structure_loss": structure,
            "train_node_loss": node, "train_edge_loss": edge, "val_f1": val_metrics["f1"],
            "val_pr_auc": val_metrics["pr_auc"], "val_roc_auc": val_metrics["roc_auc"],
        })
        print(f"{experiment['name']} | epoch {epoch:02d} | train loss {total:.4f} | validation F1 {val_metrics['f1']:.4f}")
        if val_metrics["f1"] >= best_f1:
            best_state, best_threshold, best_f1 = copy.deepcopy(model.state_dict()), threshold, val_metrics["f1"]
    model.load_state_dict(best_state)
    val_scores, val_labels = compute_anomaly_scores(model, loaders["val"], device, **weights)
    val_metrics = score_metrics(val_labels, val_scores, best_threshold)
    test_scores, test_labels = compute_anomaly_scores(model, loaders["test"], device, **weights)
    test_metrics = score_metrics(test_labels, test_scores, best_threshold)
    checkpoint = {
        "model_state_dict": best_state, "node_dim": int(bundle["node_dim"]), "edge_dim": int(bundle["edge_dim"]),
        "best_threshold": best_threshold, "edge_mean": edge_mean, "edge_std": edge_std,
        "experiment": experiment, "history": history,
    }
    return {"model": model, "checkpoint": checkpoint, "history": history, "val_metrics": val_metrics,
            "test_metrics": test_metrics, "test_scores": test_scores, "test_labels": test_labels,
            "test_predictions": (test_scores > best_threshold).astype(int), "n_train": len(train_graphs),
            "n_val": len(val_graphs), "n_test": len(test_graphs), "device": str(device)}

all_results = []
for experiment in active_experiments:
    started = time.monotonic()
    try:
        run = train_direct(experiment)
        run_id = f"{RUN_ID_PREFIX}_{experiment['name']}"
        run_dir = EXPERIMENT_ROOT / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        torch.save(run["checkpoint"], run_dir / "attribute_gae.pt")
        metrics = {
            "run_id": run_id, "dataset": DATASET, "experiment": experiment,
            "elapsed_minutes": round((time.monotonic() - started) / 60, 2),
            "n_train": run["n_train"], "n_val": run["n_val"], "n_test": run["n_test"],
            "best_threshold": run["checkpoint"]["best_threshold"], "val": run["val_metrics"],
            "test": run["test_metrics"], "history": run["history"], "device": run["device"],
        }
        (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
        all_results.append({"status": "OK", **metrics, "run_dir": str(run_dir)})
    except Exception as error:
        all_results.append({"name": experiment["name"], "status": "FAILED", "error": repr(error)})
        print(f"[{experiment['name']}] FAILED: {error}")

if not any(result["status"] == "OK" for result in all_results):
    raise RuntimeError("All direct-training experiments failed.")

baseline | epoch 01 | train loss 0.9679 | validation F1 0.8863
baseline | epoch 02 | train loss 0.8930 | validation F1 0.8845
baseline | epoch 03 | train loss 0.9077 | validation F1 0.9002
baseline | epoch 04 | train loss 0.8860 | validation F1 0.9227
baseline | epoch 05 | train loss 0.8570 | validation F1 0.8276
baseline | epoch 06 | train loss 0.8510 | validation F1 0.9251
baseline | epoch 07 | train loss 0.8501 | validation F1 0.9391
baseline | epoch 08 | train loss 0.8525 | validation F1 0.9230
baseline | epoch 09 | train loss 0.8375 | validation F1 0.9350
baseline | epoch 10 | train loss 0.8357 | validation F1 0.9328
baseline | epoch 11 | train loss 0.8329 | validation F1 0.9324
baseline | epoch 12 | train loss 0.8341 | validation F1 0.9280
baseline | epoch 13 | train loss 0.8326 | validation F1 0.9365
baseline | epoch 14 | train loss 0.8359 | validation F1 0.9254
baseline | epoch 15 | train loss 0.8299 | validation F1 0.8786
baseline | epoch 16 | train loss 0.8328 | validation F1

## 10 — Save Model Checkpoints and Metrics to Drive

In [ ]:
def sync_to_drive() -> None:
    """Copy direct-training outputs after a completed run; never train on Drive."""
    if not IN_COLAB or not EXPERIMENT_ROOT.exists():
        return
    destination = DRIVE_ROOT / "outputs" / DATASET / "gae_training_notebook"
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["rsync", "-a", "--partial", f"{EXPERIMENT_ROOT}/", f"{destination}/"])

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
summary_path = EXPERIMENT_ROOT / f"{RUN_ID_PREFIX}_summary.json"
summary_path.write_text(json.dumps(all_results, indent=2))
sync_to_drive()
print(f"Direct-training artifacts: {EXPERIMENT_ROOT}")
print(f"Run summary: {summary_path}")

## 11 — Load a Previous Direct-Training Run (Optional)

Use a saved run ID to inspect an earlier checkpoint and metrics without running training again.

In [ ]:
ANALYSIS_RUN_ID = None  # Example: "hdfs_gae_20260829_120000_baseline"

successful_results = [result for result in all_results if result["status"] == "OK"]
if ANALYSIS_RUN_ID is None:
    ANALYSIS_RUN_ID = successful_results[0]["run_id"]

local_run_dir = EXPERIMENT_ROOT / ANALYSIS_RUN_ID
if not local_run_dir.exists() and IN_COLAB:
    drive_run_dir = DRIVE_ROOT / "outputs" / DATASET / "gae_training_notebook" / ANALYSIS_RUN_ID
    if not drive_run_dir.exists():
        raise FileNotFoundError(f"No direct-training artifact directory: {drive_run_dir}")
    local_run_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["rsync", "-a", "--partial", f"{drive_run_dir}/", f"{local_run_dir}/"])
if not local_run_dir.exists():
    raise FileNotFoundError(f"No direct-training artifact directory: {local_run_dir}")

analysis_metrics = json.loads((local_run_dir / "metrics.json").read_text())
print(f"Analysing direct run: {ANALYSIS_RUN_ID}")

## 12 — Compare Direct-Training Metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

comparison_rows = []
for result in successful_results:
    val, test = result["val"], result["test"]
    comparison_rows.append({
        "run_id": result["run_id"], "name": result["experiment"]["name"],
        "duration_min": result["elapsed_minutes"], "threshold": result["best_threshold"],
        "val_f1": val["f1"], "val_pr_auc": val["pr_auc"], "val_roc_auc": val["roc_auc"],
        "test_f1": test["f1"], "test_precision": test["precision"], "test_recall": test["recall"],
        "test_pr_auc": test["pr_auc"], "test_roc_auc": test["roc_auc"],
        "n_train": result["n_train"], "n_val": result["n_val"], "n_test": result["n_test"],
    })
comparison = pd.DataFrame(comparison_rows).sort_values("test_f1", ascending=False).reset_index(drop=True)
display(comparison)
comparison.to_csv(EXPERIMENT_ROOT / f"{RUN_ID_PREFIX}_comparison.csv", index=False)
sync_to_drive()

## 13 — Plot Training and Validation Histories

In [ ]:
for result in successful_results:
    history = result["history"]
    epochs = np.asarray([entry["epoch"] for entry in history])
    figure, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    axes[0].plot(epochs, [entry["train_total_loss"] for entry in history], color="black", linewidth=2, label="total")
    axes[0].set(title="Total training loss", xlabel="Epoch", ylabel="Loss")
    for key, color in (("train_structure_loss", "#4C72B0"), ("train_node_loss", "#55A868"), ("train_edge_loss", "#C44E52")):
        axes[1].plot(epochs, [entry[key] for entry in history], linewidth=2, label=key.removeprefix("train_").removesuffix("_loss"), color=color)
    axes[1].set(title="Reconstruction components", xlabel="Epoch", ylabel="Loss")
    axes[2].plot(epochs, [entry["val_f1"] for entry in history], linewidth=2, color="#8172B2", label="Validation F1")
    axes[2].plot(epochs, [entry["val_pr_auc"] for entry in history], linewidth=1.5, color="#DD8452", label="Validation PR-AUC")
    axes[2].plot(epochs, [entry["val_roc_auc"] for entry in history], linewidth=1.5, color="#937860", label="Validation ROC-AUC")
    axes[2].set(title="Validation metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1.05))
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    figure.suptitle(result["run_id"])
    figure.tight_layout()
    figure.savefig(Path(result["run_dir"]) / "training_history.png", dpi=160, bbox_inches="tight")
    plt.show()
sync_to_drive()

## 14 — Restore a Direct Checkpoint and Decompose Test Scores

This reloads the direct-training checkpoint, applies its train-split edge normalization, and recomputes per-graph structure, node, and edge reconstruction errors on the held-out test split.

In [ ]:
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter
from src.modules.models.gae import AttributeAwareGAE

checkpoint = torch.load(local_run_dir / "attribute_gae.pt", map_location="cpu", weights_only=False)
experiment = checkpoint["experiment"]
graphs = [copy.deepcopy(graph) for graph in bundle["data_list"]]
test_graphs = [graphs[int(index)] for index in bundle["idx_test"]]
if experiment["test_run"]:
    test_graphs = test_graphs[: int(experiment["test_samples"])]

if checkpoint["edge_mean"] is not None and checkpoint["edge_std"] is not None:
    mean = torch.tensor(checkpoint["edge_mean"], dtype=torch.float32)
    std = torch.tensor(checkpoint["edge_std"], dtype=torch.float32)
    for graph in test_graphs:
        if getattr(graph, "edge_attr", None) is not None and graph.edge_attr.numel():
            graph.edge_attr = (graph.edge_attr.float() - mean) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
analysis_model = AttributeAwareGAE(
    node_dim=checkpoint["node_dim"], edge_dim=checkpoint["edge_dim"],
    hidden_dim=int(experiment["hidden_dim"]), latent_dim=int(experiment["latent_dim"]),
    gine_aggregation=experiment["gine_aggregation"], node_transformation=experiment["node_transformation"],
).to(device)
analysis_model.load_state_dict(checkpoint["model_state_dict"])
analysis_model.eval()


def component_scores(model, graphs, batch_size):
    structure_scores, node_scores, edge_scores, labels = [], [], [], []
    with torch.no_grad():
        for batch in DataLoader(graphs, batch_size=batch_size):
            batch = batch.to(device)
            z, x_norm, edge_attr = model(batch.x, batch.edge_index, batch.edge_attr)
            n_graphs = batch.num_graphs
            structure = torch.zeros(n_graphs, device=device)
            if batch.edge_index.size(1):
                logits = model.decode_structure(z, batch.edge_index)
                errors = F.binary_cross_entropy_with_logits(logits, torch.ones_like(logits), reduction="none")
                structure = scatter(errors, batch.batch[batch.edge_index[0]], dim=0, reduce="mean", dim_size=n_graphs)
            node_errors = F.mse_loss(model.decode_node_features(z), x_norm, reduction="none").mean(dim=1)
            node = scatter(node_errors, batch.batch, dim=0, reduce="mean", dim_size=n_graphs)
            edge = torch.zeros(n_graphs, device=device)
            if edge_attr is not None and edge_attr.numel():
                errors = F.mse_loss(model.decode_edge_attributes(z, batch.edge_index), edge_attr, reduction="none").mean(dim=1)
                edge = scatter(errors, batch.batch[batch.edge_index[0]], dim=0, reduce="mean", dim_size=n_graphs)
            structure_scores.append(torch.nan_to_num(structure).cpu())
            node_scores.append(torch.nan_to_num(node).cpu())
            edge_scores.append(torch.nan_to_num(edge).cpu())
            labels.append(batch.y.cpu())
    return tuple(value.numpy() for value in (
        torch.cat(structure_scores), torch.cat(node_scores), torch.cat(edge_scores), torch.cat(labels)
    ))

weights = {key: float(experiment[key]) for key in ("alpha", "beta", "gamma")}
test_structure, test_node, test_edge, test_labels = component_scores(analysis_model, test_graphs, int(experiment["batch_size"]))
test_scores = weights["alpha"] * test_structure + weights["beta"] * test_node + weights["gamma"] * test_edge
threshold = float(checkpoint["best_threshold"])
test_predictions = (test_scores > threshold).astype(int)
print(f"Re-evaluated {len(test_labels):,} held-out graphs from {ANALYSIS_RUN_ID}.")

## 15 — Test Curves, Score Distributions, and Contribution Analysis

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve,
)

FIGURE_DIR = local_run_dir / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
print(classification_report(test_labels, test_predictions, digits=4, zero_division=0))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, color, title in ((0, "#4C72B0", "Normal"), (1, "#C44E52", "Anomaly")):
    subset = test_scores[test_labels == label]
    if len(subset):
        axes[0].hist(subset, density=True, bins=40, alpha=0.45, color=color, label=f"{title} (n={len(subset)})")
axes[0].axvline(threshold, color="black", linestyle="--", label="validation threshold")
axes[0].set(title="Held-out anomaly-score distribution", xlabel="Weighted reconstruction error")
axes[0].legend()
ConfusionMatrixDisplay(confusion_matrix(test_labels, test_predictions, labels=[0, 1]), display_labels=["Normal", "Anomaly"]).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Held-out confusion matrix")
figure.tight_layout()
figure.savefig(FIGURE_DIR / "test_distribution_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

if len(np.unique(test_labels)) == 2:
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    precision, recall, _ = precision_recall_curve(test_labels, test_scores)
    axes[0].plot(recall, precision, linewidth=2, label=f"PR-AUC = {average_precision_score(test_labels, test_scores):.4f}")
    axes[0].axhline(test_labels.mean(), color="gray", linestyle="--", label="positive-class rate")
    axes[0].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
    axes[0].legend()
    false_positive_rate, true_positive_rate, _ = roc_curve(test_labels, test_scores)
    axes[1].plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"ROC-AUC = {roc_auc_score(test_labels, test_scores):.4f}")
    axes[1].plot([0, 1], [0, 1], "--", color="gray", label="random")
    axes[1].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="False positive rate", ylabel="True positive rate", title="ROC curve")
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "test_pr_roc.png", dpi=160, bbox_inches="tight")
    plt.show()

components = pd.DataFrame({
    "Structure": weights["alpha"] * test_structure, "Node": weights["beta"] * test_node,
    "Edge": weights["gamma"] * test_edge, "label": test_labels, "prediction": test_predictions,
})
true_positives = components[(components.label == 1) & (components.prediction == 1)].copy()
if len(true_positives):
    contribution = true_positives[["Structure", "Node", "Edge"]].div(true_positives[["Structure", "Node", "Edge"]].sum(axis=1), axis=0).fillna(0)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    average_contribution = contribution.mean().mul(100)
    axes[0].pie(average_contribution, labels=average_contribution.index, autopct="%1.1f%%")
    axes[0].set_title(f"Average weighted contribution — true positives (n={len(true_positives)})")
    dominant = contribution.idxmax(axis=1).value_counts().reindex(["Structure", "Node", "Edge"], fill_value=0)
    axes[1].bar(dominant.index, dominant.values, color=["#4C72B0", "#55A868", "#C44E52"])
    axes[1].set(title="Dominant component per true positive", ylabel="Graphs")
    axes[1].grid(axis="y", alpha=0.25)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "component_contribution.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("No true positives are available for component-contribution analysis.")
components.to_csv(FIGURE_DIR / "test_component_scores.csv", index=False)
sync_to_drive()

## 16 — Split-Leakage Checks and Final Artifact Export

The checks below inspect the saved graph bundle and the restored checkpoint. They verify split identities where available and confirm that clean training retained normal graphs only.

In [ ]:
def split_graphs(index_key: str) -> list:
    return [bundle["data_list"][int(index)] for index in bundle[index_key]]


def graph_ids(graphs: list) -> set[str]:
    identifiers = set()
    for graph in graphs:
        for attribute in ("block_id", "sequence_id", "graph_id"):
            if hasattr(graph, attribute):
                value = getattr(graph, attribute)
                identifiers.add(str(value.item() if hasattr(value, "item") else value))
                break
    return identifiers


raw_train_graphs = split_graphs("idx_train")
val_graphs = split_graphs("idx_val")
original_test_graphs = split_graphs("idx_test")
trained_graphs = raw_train_graphs if experiment["train_mode"] == "noisy" else [
    graph for graph in raw_train_graphs if int(graph.y.item()) == 0
]
train_ids, val_ids, held_out_ids = map(graph_ids, (raw_train_graphs, val_graphs, original_test_graphs))
print("=== Data leakage verification ===")
if train_ids or val_ids or held_out_ids:
    print(f"Train–validation identifier overlap: {len(train_ids & val_ids)}")
    print(f"Train–test identifier overlap: {len(train_ids & held_out_ids)}")
    print(f"Validation–test identifier overlap: {len(val_ids & held_out_ids)}")
    assert not (train_ids & val_ids or train_ids & held_out_ids or val_ids & held_out_ids), "Split identifiers overlap."
    print("Identifier-disjoint splits: PASS")
else:
    print("No stable graph identifier field is stored; identity-overlap check is unavailable.")


for name, graphs in (("train source", raw_train_graphs), ("train used", trained_graphs), ("validation", val_graphs), ("test", original_test_graphs)):
    labels = np.asarray([int(graph.y.item()) for graph in graphs])
    print(f"{name:12}: {len(labels):,} graphs; anomalies={labels.sum():,}; rate={labels.mean() if len(labels) else 0:.4f}")
if experiment["train_mode"] == "clean":
    assert all(int(graph.y.item()) == 0 for graph in trained_graphs)
    print("Clean training contains no anomalous graphs: PASS")
else:
    print("Noisy training intentionally retains the original training split.")


batch_norm = analysis_model.raw_node_norm
summary = {
    "analysis_run_id": ANALYSIS_RUN_ID,
    "threshold": threshold,
    "n_recomputed_test": int(len(test_labels)),
    "test_positive_rate": float(np.mean(test_labels)),
    "batch_norm_running_mean_first5": batch_norm.running_mean[:5].detach().cpu().tolist(),
    "batch_norm_running_var_first5": batch_norm.running_var[:5].detach().cpu().tolist(),
}
summary_path = FIGURE_DIR / f"{ANALYSIS_RUN_ID}_analysis_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
sync_to_drive()
print(f"Analysis summary saved to {summary_path}")